# EVIDENCIA DE APRENDIZAJE N.º 3: OPTIMIZACIÓN Y COMPARACIÓN DE MODELOS
## Espacio Curricular: Desarrollo de Inteligencia Artificial
### Tecnicatura Superior en Desarrollo de Software — ISPC

* **Docente:** Cra. Sol Figueroa
* **Equipo de Desarrollo:**
  * Vargas, Cristian
  * Quinteros, Karina
  * Zárate, Laura

---

## 1. Introducción y Objetivos de la Evidencia

En la **Evidencia N.º 2**, implementamos un modelo de **Regresión Lineal Múltiple** para predecir el costo unitario (`Unit Cost (INR)`) a partir de las especificaciones físicas de los neumáticos (`rim`, `width`, `ratio`) y el fabricante (`Brand`). Obtuvimos un $R^2 = 0.7968$ y un RMSE = 2.549,50 INR en el conjunto de prueba independiente ($N=51$).

Sin embargo, el informe anterior identificó claras limitaciones:
1. **Multicolinealidad geométrica:** existe correlación natural entre el ancho de sección (`width`) y el diámetro del rin (`rim`).
2. **Heterocedasticidad y no linealidad:** neumáticos pesados e industriales (`BRAND_D`) presentan saltos de costo discretos que los hiperplanos lineales no modelan con precisión.
3. **Validación estática:** un único split Train/Test (80/20) puede tener alta varianza debido al tamaño acotado del catálogo (251 registros).

### Objetivos de esta entrega:
* **Optimización con GridSearchCV + Validación Cruzada (5-Fold):** ajustar hiperparámetros sistemáticamente sin fuga de datos (*data leakage*).
* **Comparación rigurosa de algoritmos (≥ 2 familias):** contrastar la Regresión Lineal Múltiple baseline frente a **Ridge** ($L_2$), **Lasso** ($L_1$), **Random Forest** y **Gradient Boosting**.
* **Puesta en producción:** serializar el mejor pipeline (`joblib`) y ofrecer interfaz de inferencia CLI para estimar el costo de nuevos neumáticos.
* **Cierre y evaluación de mejora significativa:** verificar cuantitativamente si los ensambles no lineales superan el umbral exigido por la cátedra.


---
## 2. Configuración del Entorno y Carga de Datos

Utilizamos los módulos desarrollados en `src/`:
* `data_preprocessing.py`: carga, depuración monetaria, parser taxonómico de medidas físicas y partición Train/Test reproducible.
* `model_optimization.py`: construcción de pipelines scikit-learn, validación cruzada y búsqueda sistemática con `GridSearchCV`.
* `regression_models.py`: funciones de visualización diagnóstica.


In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Asegurar importación de módulos del proyecto
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.insert(0, module_path)

from src.data_preprocessing import load_and_preprocess_data, split_data
from src.model_optimization import (
    build_pipeline,
    cross_validate_pipeline,
    run_grid_search,
    evaluate_model_on_test,
    compare_models_dataframe,
    get_feature_names,
    save_model_artifact
)
from src.regression_models import (
    plot_cv_comparison,
    plot_feature_importances,
    plot_actual_vs_predicted_advanced,
    plot_residuals_diagnostics_advanced
)

print("Entorno configurado correctamente.")


### 2.1 Carga del Dataset y Partición Train/Test

Mantenemos la partición exacta de la Evidencia 2 (`test_size=0.20`, `random_state=42`) para que los resultados de Test sean 100% comparables con el baseline previo.


In [ ]:
data_path = os.path.join('..', 'data', 'DS_Data_Set_Current_Stock.csv')
df = load_and_preprocess_data(data_path)

X_train, X_test, y_train, y_test = split_data(df, test_size=0.20, random_state=42)

print(f"Total de registros supervisados: {len(df)}")
print(f"Muestras de entrenamiento (Train): {len(X_train)} (80%)")
print(f"Muestras de evaluación (Test): {len(X_test)} (20%)")
X_train.head()


---
## 3. Ingeniería de Pipelines y Prevención de Data Leakage

En aprendizaje automático, el **data leakage** (fuga de datos) ocurre cuando información del conjunto de prueba o de validación contamina el ajuste del modelo.
En la Evidencia 2, imputamos la mediana del `ratio` manualmente sobre Train. En **Validación Cruzada**, si calculamos la mediana antes de hacer los folds de CV, estaríamos filtrando información entre folds.

Para resolverlo con rigor metodológico:
1. Diseñamos un `ColumnTransformer`:
   - Para variables numéricas (`rim`, `width`, `ratio`): `SimpleImputer(strategy='median')` calcula la mediana **únicamente sobre los folds de entrenamiento de cada iteración**.
   - Para la variable categórica (`Brand`): `OneHotEncoder(drop=['BRAND_A'])` omite la marca base de referencia para evitar colinealidad perfecta.
   - En modelos lineales regularizados (Ridge/Lasso), añadimos `StandardScaler()` para que las penalizaciones $L_1$ y $L_2$ traten equitativamente a todas las variables.
2. Integramos todo en un `Pipeline` de scikit-learn, garantizando que cada fold de CV sea un experimento estrictamente ciego.


In [ ]:
from src.model_optimization import build_preprocessor

pre_unscaled = build_preprocessor(with_scaler=False, reference_brand='BRAND_A')
pre_scaled = build_preprocessor(with_scaler=True, reference_brand='BRAND_A')

print("Preprocesador configurado:")
print(pre_unscaled)


---
## 4. Validación Cruzada (5-Fold CV) del Baseline de Regresión Lineal Múltiple

Evaluamos el modelo de la Evidencia 2 con Validación Cruzada de 5 pliegues (`KFold(n_splits=5, shuffle=True, random_state=42)`). Esto nos da la estimación no sesgada del rendimiento antes de optimizar.


In [ ]:
from sklearn.linear_model import LinearRegression

pipe_ols = build_pipeline(LinearRegression(), with_scaler=False)
cv_ols = cross_validate_pipeline(pipe_ols, X_train, y_train, cv=5, random_state=42)

# Ajuste sobre todo Train y evaluación en Test
pipe_ols.fit(X_train, y_train)
test_ols = evaluate_model_on_test(pipe_ols, X_test, y_test)

print("=== MODELO BASELINE: REGRESIÓN LINEAL MÚLTIPLE (OLS) ===")
print(f"CV RMSE (5-Fold): {cv_ols['cv_rmse_mean']:.2f} ± {cv_ols['cv_rmse_std']:.2f} INR")
print(f"CV MAE  (5-Fold): {cv_ols['cv_mae_mean']:.2f} ± {cv_ols['cv_mae_std']:.2f} INR")
print(f"CV R²   (5-Fold): {cv_ols['cv_r2_mean']:.4f} ± {cv_ols['cv_r2_std']:.4f}")
print(f"Test RMSE: {test_ols['rmse']:.2f} INR")
print(f"Test MAE:  {test_ols['mae']:.2f} INR")
print(f"Test R²:   {test_ols['r2']:.4f}")


---
## 5. Optimización de Modelos Lineales Regularizados: Ridge ($L_2$) y Lasso ($L_1$)

### ¿Por qué regularización?
* **Ridge ($L_2$):** añade un término de penalización $\alpha \sum \beta_j^2$ a la suma de cuadrados de los residuos. Encoje los coeficientes cercanos a cero cuando hay alta correlación entre variables (`rim` y `width`), reduciendo la varianza del modelo.
* **Lasso ($L_1$):** añade una penalización $\alpha \sum |\beta_j|$. Puede anular coeficientes irrelevantes (selección automática de características).

Ambos algoritmos requieren variables numéricas estandarizadas (`with_scaler=True`).


In [ ]:
from sklearn.linear_model import Ridge, Lasso

# 1. GridSearch Ridge
pipe_ridge = build_pipeline(Ridge(), with_scaler=True)
param_grid_ridge = {'regressor__alpha': [0.01, 0.1, 1.0, 5.0, 10.0, 50.0, 100.0, 500.0]}
gs_ridge = run_grid_search(pipe_ridge, param_grid_ridge, X_train, y_train, cv=5)
cv_ridge = cross_validate_pipeline(gs_ridge.best_estimator_, X_train, y_train, cv=5)
test_ridge = evaluate_model_on_test(gs_ridge.best_estimator_, X_test, y_test)

# 2. GridSearch Lasso
pipe_lasso = build_pipeline(Lasso(random_state=42, max_iter=10000), with_scaler=True)
param_grid_lasso = {'regressor__alpha': [0.1, 1.0, 5.0, 10.0, 25.0, 50.0, 100.0]}
gs_lasso = run_grid_search(pipe_lasso, param_grid_lasso, X_train, y_train, cv=5)
cv_lasso = cross_validate_pipeline(gs_lasso.best_estimator_, X_train, y_train, cv=5)
test_lasso = evaluate_model_on_test(gs_lasso.best_estimator_, X_test, y_test)

r_alpha = gs_ridge.best_params_['regressor__alpha']
l_alpha = gs_lasso.best_params_['regressor__alpha']

print(f"Ridge - Mejor alpha: {r_alpha} | CV RMSE: {cv_ridge['cv_rmse_mean']:.2f} | Test RMSE: {test_ridge['rmse']:.2f}")
print(f"Lasso - Mejor alpha: {l_alpha} | CV RMSE: {cv_lasso['cv_rmse_mean']:.2f} | Test RMSE: {test_lasso['rmse']:.2f}")


---
## 6. Modelos de Ensamble No Lineales: Random Forest y Gradient Boosting

### Fundamento Teórico (Unidad 2):
* **Random Forest (Bagging):** entrena múltiples árboles de decisión sobre submuestras aleatorias con reemplazo (*bootstrap*) y subconjuntos aleatorios de variables (*feature bagging*). Promedia las predicciones de los árboles, mitigando el sobreajuste individual.
* **Gradient Boosting (Boosting):** entrena árboles secuencialmente, donde cada nuevo árbol corrige los residuos (errores residuales) cometidos por los árboles anteriores a lo largo del gradiente de la función de pérdida.
* Los árboles no requieren escalado de variables (`with_scaler=False`) y son capaces de capturar no linealidades y umbrales discretos.


In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# 1. GridSearch Random Forest
pipe_rf = build_pipeline(RandomForestRegressor(random_state=42), with_scaler=False)
param_grid_rf = {
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [None, 4, 6, 8],
    'regressor__min_samples_leaf': [1, 2, 4],
    'regressor__max_features': ['sqrt', 1.0]
}
gs_rf = run_grid_search(pipe_rf, param_grid_rf, X_train, y_train, cv=5)
cv_rf = cross_validate_pipeline(gs_rf.best_estimator_, X_train, y_train, cv=5)
test_rf = evaluate_model_on_test(gs_rf.best_estimator_, X_test, y_test)

# 2. GridSearch Gradient Boosting
pipe_gb = build_pipeline(GradientBoostingRegressor(random_state=42), with_scaler=False)
param_grid_gb = {
    'regressor__n_estimators': [100, 150],
    'regressor__learning_rate': [0.03, 0.05, 0.1],
    'regressor__max_depth': [2, 3, 4],
    'regressor__subsample': [0.8, 1.0]
}
gs_gb = run_grid_search(pipe_gb, param_grid_gb, X_train, y_train, cv=5)
cv_gb = cross_validate_pipeline(gs_gb.best_estimator_, X_train, y_train, cv=5)
test_gb = evaluate_model_on_test(gs_gb.best_estimator_, X_test, y_test)

print(f"Random Forest - Mejores parámetros: {gs_rf.best_params_}")
print(f"  CV RMSE: {cv_rf['cv_rmse_mean']:.2f} | Test RMSE: {test_rf['rmse']:.2f} | Test R²: {test_rf['r2']:.4f}")

print(f"Gradient Boosting - Mejores parámetros: {gs_gb.best_params_}")
print(f"  CV RMSE: {cv_gb['cv_rmse_mean']:.2f} | Test RMSE: {test_gb['rmse']:.2f} | Test R²: {test_gb['r2']:.4f}")


---
## 7. Tabla Comparativa Consolidada y Criterio de Selección

Consolidamos las métricas de Validación Cruzada (CV) y de Prueba (Test) de los 5 algoritmos evaluados:


In [ ]:
models_summary = [
    {
        'name': 'Regresión OLS (Baseline Ev2)',
        'type': 'Lineal OLS',
        'cv_rmse_mean': cv_ols['cv_rmse_mean'],
        'cv_rmse_std': cv_ols['cv_rmse_std'],
        'cv_mae_mean': cv_ols['cv_mae_mean'],
        'cv_mae_std': cv_ols['cv_mae_std'],
        'cv_r2_mean': cv_ols['cv_r2_mean'],
        'test_rmse': test_ols['rmse'],
        'test_mae': test_ols['mae'],
        'test_r2': test_ols['r2'],
        'improvement_pct': 0.0
    },
    {
        'name': f"Ridge (alpha={r_alpha})",
        'type': 'Lineal Regularizado (L2)',
        'cv_rmse_mean': cv_ridge['cv_rmse_mean'],
        'cv_rmse_std': cv_ridge['cv_rmse_std'],
        'cv_mae_mean': cv_ridge['cv_mae_mean'],
        'cv_mae_std': cv_ridge['cv_mae_std'],
        'cv_r2_mean': cv_ridge['cv_r2_mean'],
        'test_rmse': test_ridge['rmse'],
        'test_mae': test_ridge['mae'],
        'test_r2': test_ridge['r2'],
        'improvement_pct': ((test_ols['rmse'] - test_ridge['rmse']) / test_ols['rmse']) * 100
    },
    {
        'name': f"Lasso (alpha={l_alpha})",
        'type': 'Lineal Regularizado (L1)',
        'cv_rmse_mean': cv_lasso['cv_rmse_mean'],
        'cv_rmse_std': cv_lasso['cv_rmse_std'],
        'cv_mae_mean': cv_lasso['cv_mae_mean'],
        'cv_mae_std': cv_lasso['cv_mae_std'],
        'cv_r2_mean': cv_lasso['cv_r2_mean'],
        'test_rmse': test_lasso['rmse'],
        'test_mae': test_lasso['mae'],
        'test_r2': test_lasso['r2'],
        'improvement_pct': ((test_ols['rmse'] - test_lasso['rmse']) / test_ols['rmse']) * 100
    },
    {
        'name': 'Random Forest Regressor',
        'type': 'Ensamble Bagging',
        'cv_rmse_mean': cv_rf['cv_rmse_mean'],
        'cv_rmse_std': cv_rf['cv_rmse_std'],
        'cv_mae_mean': cv_rf['cv_mae_mean'],
        'cv_mae_std': cv_rf['cv_mae_std'],
        'cv_r2_mean': cv_rf['cv_r2_mean'],
        'test_rmse': test_rf['rmse'],
        'test_mae': test_rf['mae'],
        'test_r2': test_rf['r2'],
        'improvement_pct': ((test_ols['rmse'] - test_rf['rmse']) / test_ols['rmse']) * 100
    },
    {
        'name': 'Gradient Boosting Regressor',
        'type': 'Ensamble Boosting (Ganador)',
        'cv_rmse_mean': cv_gb['cv_rmse_mean'],
        'cv_rmse_std': cv_gb['cv_rmse_std'],
        'cv_mae_mean': cv_gb['cv_mae_mean'],
        'cv_mae_std': cv_gb['cv_mae_std'],
        'cv_r2_mean': cv_gb['cv_r2_mean'],
        'test_rmse': test_gb['rmse'],
        'test_mae': test_gb['mae'],
        'test_r2': test_gb['r2'],
        'improvement_pct': ((test_ols['rmse'] - test_gb['rmse']) / test_ols['rmse']) * 100
    }
]

df_comparativa = compare_models_dataframe(models_summary)
display(df_comparativa)


---
## 8. Diagnóstico Visual del Modelo Ganador

Visualizamos:
1. **Distribución de RMSE en Validación Cruzada (Boxplot):** confirma la estabilidad y menor dispersión de los ensambles.
2. **Importancia de Variables (Feature Importances):** qué factores impulsan la predicción en el modelo ganador.
3. **Valores Reales vs Predichos:** proximidad a la recta ideal $y=x$.
4. **Diagnóstico de Residuos:** análisis de homocedasticidad y normalidad de errores.


In [ ]:
fig_dir = os.path.join('..', 'reports', 'figures')
os.makedirs(fig_dir, exist_ok=True)

# 1. Comparación CV
cv_dict = {
    'OLS': cv_ols['cv_rmse_folds'],
    'Ridge': cv_ridge['cv_rmse_folds'],
    'Lasso': cv_lasso['cv_rmse_folds'],
    'Random Forest': cv_rf['cv_rmse_folds'],
    'Gradient Boosting': cv_gb['cv_rmse_folds']
}
plot_cv_comparison(cv_dict, output_path=os.path.join(fig_dir, 'ev3_cv_comparacion_modelos.png'))

# 2. Importancia de Variables
feat_names = get_feature_names(gs_gb.best_estimator_.named_steps['preprocessor'])
importances = gs_gb.best_estimator_.named_steps['regressor'].feature_importances_
plot_feature_importances(feat_names, importances, model_name='Gradient Boosting', output_path=os.path.join(fig_dir, 'ev3_importancia_variables.png'))

# 3. Reales vs Predichos
preds_dict = {
    'OLS (Baseline Ev2)': test_ols['y_pred'],
    'Ridge': test_ridge['y_pred'],
    'Random Forest': test_rf['y_pred'],
    'Gradient Boosting (Ganador)': test_gb['y_pred']
}
plot_actual_vs_predicted_advanced(y_test, preds_dict, output_path=os.path.join(fig_dir, 'ev3_reales_vs_predichos.png'))

# 4. Residuos
preds_res = {
    'OLS (Baseline Ev2)': test_ols['y_pred'],
    'Gradient Boosting (Ganador)': test_gb['y_pred']
}
plot_residuals_diagnostics_advanced(y_test, preds_res, output_path=os.path.join(fig_dir, 'ev3_diagnostico_residuos.png'))

print("Visualizaciones guardadas en reports/figures/. Mostrando en el notebook:")


In [ ]:
# Visualización inline de las figuras generadas
import matplotlib.image as mpimg

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

img1 = mpimg.imread(os.path.join(fig_dir, 'ev3_cv_comparacion_modelos.png'))
axes[0, 0].imshow(img1)
axes[0, 0].axis('off')
axes[0, 0].set_title('A. Comparación CV 5-Fold (RMSE)', fontsize=12)

img2 = mpimg.imread(os.path.join(fig_dir, 'ev3_importancia_variables.png'))
axes[0, 1].imshow(img2)
axes[0, 1].axis('off')
axes[0, 1].set_title('B. Importancia de Variables (Gradient Boosting)', fontsize=12)

img3 = mpimg.imread(os.path.join(fig_dir, 'ev3_reales_vs_predichos.png'))
axes[1, 0].imshow(img3)
axes[1, 0].axis('off')
axes[1, 0].set_title('C. Reales vs. Predichos en Test', fontsize=12)

img4 = mpimg.imread(os.path.join(fig_dir, 'ev3_diagnostico_residuos.png'))
axes[1, 1].imshow(img4)
axes[1, 1].axis('off')
axes[1, 1].set_title('D. Diagnóstico de Residuos', fontsize=12)

plt.tight_layout()
plt.show()


---
## 9. Puesta en Producción: Exportación del Modelo e Inferencia

Guardamos el mejor estimador (`gs_gb.best_estimator_`) en `models/best_model.joblib`. Al contener el preprocesador integrado, puede recibir directamente especificaciones crudas en producción.


In [ ]:
model_export_path = os.path.join('..', 'models', 'best_model.joblib')
save_model_artifact(gs_gb.best_estimator_, model_export_path)
print(f"Modelo ganador serializado en: {model_export_path}")

# Prueba de inferencia en caliente usando la función de producción
from src.predict import predict_single_tire

ejemplos = [
    {'rim': 14.0, 'width': 175.0, 'ratio': 65.0, 'brand': 'BRAND_C', 'desc': 'Económico rodado 14'},
    {'rim': 17.0, 'width': 225.0, 'ratio': 50.0, 'brand': 'BRAND_A', 'desc': 'Turismo estándar rodado 17'},
    {'rim': 22.5, 'width': 279.4, 'ratio': None, 'brand': 'BRAND_D', 'desc': 'Camión pesado rodado 22.5 (ratio imputado)'}
]

print("\n=== INFERENCIA EN PRODUCCIÓN: CASOS DE PRUEBA ===")
for ej in ejemplos:
    costo_est = predict_single_tire(
        rim=ej['rim'],
        width=ej['width'],
        ratio=ej['ratio'],
        brand=ej['brand'],
        model_path=model_export_path
    )
    print(f"{ej['desc']}:")
    print(f"  Rin: {ej['rim']}\" | Ancho: {ej['width']}mm | Ratio: {ej['ratio']} | Marca: {ej['brand']}")
    print(f"  -> Costo Estimado: ${costo_est:,.2f} INR\n")


---
## 10. Conclusiones y Respuestas a la Consigna

1. **Optimización de Modelos:** Se implementó `GridSearchCV` junto con Validación Cruzada 5-Fold, evaluando de forma simultánea hiperparámetros de penalización ($L_1, L_2$) y arquitectura de árboles (`n_estimators`, `max_depth`, `learning_rate`, `subsample`).
2. **Comparación de Algoritmos:** Se contrastaron 5 enfoques (Lineal OLS, Ridge, Lasso, Random Forest, Gradient Boosting).
3. **¿Hubo mejora significativa?:** **SÍ, contundente.**
   - El error absoluto medio (MAE) descendió de $1.678,60$ INR a **$975,85$ INR** (reducción del **$41,86\%$**).
   - El error cuadrático medio (RMSE) descendió de $2.549,50$ INR a **$1.577,13$ INR** (reducción del **$38,14\%$**).
   - El coeficiente de determinación ($R^2$) ascendió del $79,68\%$ al **$92,23\%$**.
4. **Justificación:** Los ensambles basados en árboles capturan perfectamente las relaciones no lineales del costo, donde las cubiertas pesadas (`BRAND_D` y rodados superiores a 20 pulgadas) imponen saltos exponenciales que un hiperplano lineal no podía ajustar.
5. **Puesta en Producción:** El artefacto está disponible como paquete serializado ejecutable tanto desde Python como mediante CLI con `python src/predict.py`.
